In [1]:
import sys, os
sys.path.append(os.getcwd().split('src')[0] + 'src')
from utils import *
from simulations.src import *

In [2]:
manifold_type = 'S1'; manifold = get_manifold(manifold_type)

# n_samples_ls = [100, 500, 1000, 2500, 5000, 7500, 10000,25000, 50000]
n_samples_ls =  [100, 500, 1000, 2500, 5000, 7500, 10000]

manifold_type = 'S1'

G_sampler_ls = [
    get_G_class(manifold_type, sampler, name, params) for sampler, name, params 
        in [
            (multimodal_sampler, '1-modal', {'tau2' : 0.1, 'num_modes' : 1}),
            (multimodal_sampler, '2-modal', {'tau2' : 0.075, 'num_modes' : 2}),
            (multimodal_sampler, '3-modal', {'tau2' : 0.05, 'num_modes' : 3}),
            (multimodal_sampler, '4-modal', {'tau2' : 0.025, 'num_modes' : 4}),
        ]
    ]
sigma2 = 0.1

NMC = 2

test_size = 10
num_oracle_samples = 10

M_grid= np.arange(1, 9)
rho_grid = [0.015, 0.025, 0.045, 0.065, 0.085, 0.10, 0.125, 0.15, 0.175, 0.2]


In [ ]:
timenow = time.time()
results_mc = []
results_ocv = []
for G in G_sampler_ls:
    dfmc, dforaclecv = converenge_rate_experiment(manifold_type, G, n_samples_ls, M_grid, rho_grid, sigma2, test_size, num_oracle_samples, NMC, timenow, cv=True)
    results_mc.append(dfmc)
    results_ocv.append(dforaclecv)
    
results_mc_ = pd.concat(results_mc, ignore_index=True)
results_ocv_ = pd.concat(results_ocv, ignore_index=True)

params_ = {
    'ID' : timenow,
    'manifold_type' : manifold_type,
    'n_samples_ls' : n_samples_ls,
    'M_grid' : M_grid,
    'rho_grid' : rho_grid,
    'sigma2' : sigma2,
    'test_size' : test_size,
    'num_oracle_samples' : num_oracle_samples,
    'NMC' : NMC,
    'G_names' : [G.name for G in G_sampler_ls],
    'G_params' : [G.params for G in G_sampler_ls]
}

for name, results in zip(['mc', 'ocv', 'params'], [results_mc_, results_ocv_, params_]):

    if name == 'params':
        filepath = 'data/{}/rate_{}_{}_{}.csv'.format(manifold_type,manifold_type,name,timenow)
        pickle.dump(results, open(filepath, 'wb'))
    else:
        filepath = 'data/{}/rate_{}_{}.csv'.format(manifold_type,manifold_type,name)
        if os.path.exists(filepath):
            print(f"'{filepath}' already exists.")
            choice = input("Choose an option — [c] concatenate, [o] overwrite, [n] do nothing: ").strip().lower()
            if choice == 'c':
                existing = pd.read_csv(filepath)
                combined = pd.concat([existing, results], ignore_index=True).drop_duplicates()
                combined.to_csv(filepath, index=False)
                print(f"Files concatenated and saved ({len(combined)} rows, duplicates removed).")
            elif choice == 'o':
                results.to_csv(filepath, index=False)
                print("File overwritten.")
            elif choice == 'n':
                print("No changes made.")
            else:
                print("Invalid input. No changes made.")
        else:
            results.to_csv(filepath, index=False)
            print(f"File saved to '{filepath}'.")


G="1-modal", σ²=0.1:   0%|          | 0/1120 [00:00<?, ?it/s]

G="2-modal", σ²=0.1:   0%|          | 0/1120 [00:00<?, ?it/s]

G="3-modal", σ²=0.1:   0%|          | 0/1120 [00:00<?, ?it/s]

G="4-modal", σ²=0.1:   0%|          | 0/1120 [00:00<?, ?it/s]

NameError: name 'filepath' is not defined

-----

In [19]:
# List files in data/{manifold_type}/ containing "params" in the filename
data_dir = os.path.join("data", manifold_type)
params_files = sorted(
    f for f in os.listdir(data_dir)
    if "params" in f and os.path.isfile(os.path.join(data_dir, f))
)
print([file.split('_')[-1].split('.csv')[0] for file in params_files])

['1772791544.698209']


In [21]:
ID = 1772791544.698209

results_ocv = pd.read_csv(f'data/{manifold_type}/rate_{manifold_type}_ocv.csv')
results_mc = pd.read_csv(f'data/{manifold_type}/rate_{manifold_type}_mc.csv')

In [26]:
# Execution
plot_cv_distributions_split(results_ocv, ID= ID, selected_sigma2 = sigma2)

AttributeError: 'DataFrame' object has no attribute 'ID'

In [39]:
# Snap the theoretical choices to the nearest available grid values (and keep them in-range)
extselected_Mrho = {}

M_grid_arr = np.asarray(M_grid, dtype=float)
rho_grid_arr = np.asarray(rho_grid, dtype=float)

for G in G_sampler_ls:
    n_arr = np.asarray(n_samples_ls, dtype=float)

    # original (continuous) rules
    rho_raw =  .5*(n_arr ** (-1/5))
    M_raw = np.log(n_arr) - 1

    # nearest grid points
    rho_snap = rho_grid_arr[np.argmin(np.abs(rho_raw[:, None] - rho_grid_arr[None, :]), axis=1)]
    M_snap = M_grid_arr[np.argmin(np.abs(M_raw[:, None] - M_grid_arr[None, :]), axis=1)].astype(int)


    rho_snap =  [0.015]*len(n_arr)#.5*(n_arr ** (-1/5))
    M_snap = [4,5,6,6,7,8,8]#np.log(n_arr) - 1


    extselected_Mrho[G.name] = {"rho": rho_snap, "M": M_snap}

In [5]:
np.log(n_arr)

NameError: name 'n_arr' is not defined

In [ ]:
plot_mcratesims_interactive(manifold_type, results_mc, results_ocv, G_sampler_ls, NMC, extselected_Mrho )

Output()

-----

-----

-----